# newbench_27 — Residue-Fill Review + Overview via PyMOL

**Runs on CESGA via `juplaunch`**  (memory `[[reference_juplaunch]]`).

## Connection setup

On your **laptop** before launching this notebook:

```bash
# 1. Start PyMOL with the RPC server on the laptop
export PYMOL_PATH=/usr/lib/python3/dist-packages/pymol   # Debian-fix per feedback_pymol_remote_debian
pymol -R
# → PyMOL RPC listening on :9123

# 2. Set up the reverse tunnel to CESGA
ssh -R 9123:localhost:9123 ft3.cesga.es
```

On CESGA:

```bash
~/bin/juplaunch     # prints the JupyterLab URL to paste into your laptop browser
```

Then open this notebook in the browser and run the cells top-down.
The `PymolSession(hostname="localhost", port=9123)` call reaches through
the reverse tunnel to the laptop's PyMOL GUI.


## Step 1 — Connect to laptop PyMOL

In [ ]:
# Connect to laptop PyMOL via reverse tunnel (localhost:9123)
from pymol_remote.client import PymolSession
import sys, pathlib

pm = PymolSession(hostname="localhost", port=9123, timeout=10.0)
print("Connected. Objects currently loaded:", pm.get_names())


## Step 2 — Discover delivered newbench_27 proteins

In [ ]:
# Discover delivered newbench_27 proteins
import pathlib, json

DELIVERED_ROOT = pathlib.Path("/mnt/netapp1/Store_othcxlwa/newbench_27/delivered")
BENCH_ROOT     = pathlib.Path("/mnt/netapp1/Store_othcxlwa/newbench_27")

pdb_ids = sorted(p.name for p in DELIVERED_ROOT.iterdir() if p.is_dir())
print(f"{len(pdb_ids)} delivered newbench_27 proteins:")
print(" ".join(pdb_ids))


## Step 3 — Per-protein summary (KI-Aufteilung, Masse, Actives)

Parses every `MANIFEST.json` + `graft/*_graft_manifest.json` under `delivered/` and builds one row per protein with the AI source (AF-splice vs BL-Pose fallback), estimated molecular weight, gap counts, and any warnings the pipeline logged.

In [ ]:
# Parse every delivered protein's manifest + graft_manifest and build a
# per-PDB summary dataframe.
import json, pathlib, re
import pandas as pd

def _residue_count(pdb_path: pathlib.Path) -> int:
    seen = set()
    if not pdb_path.is_file():
        return 0
    for line in pdb_path.read_text(errors='replace').splitlines():
        if line.startswith('ATOM') and len(line) > 26:
            key = (line[21], line[22:27].strip())  # chain + resnum+icode
            seen.add(key)
    return len(seen)

# Approximate mean AA molecular weight (Da) for quick MW estimate
_AVG_AA_MW = 110.0

def summarise(pdb_id: str) -> dict:
    entry = DELIVERED_ROOT / pdb_id
    manifest = {}
    if (entry / 'MANIFEST.json').is_file():
        manifest = json.loads((entry / 'MANIFEST.json').read_text())
    graft = {}
    gp = entry / 'graft' / f'{pdb_id}_graft_manifest.json'
    if gp.is_file():
        graft = json.loads(gp.read_text())
    receptor = entry / 'receptor.pdb'
    n_res = _residue_count(receptor) if receptor.is_file() else 0
    mw_kda = round(n_res * _AVG_AA_MW / 1000.0, 1)
    variant = manifest.get('variant', '—')
    method = graft.get('method', '') or ''
    # AI split heuristic — primary key = MANIFEST.json 'variant' field:
    #   'single'              → no fill needed (crystal is complete)
    #   'large_gap_graft'     → BL-Pose Kabsch graft (no AF)
    #   'large_gap_complete'  → AF-splice + MODELLER LoopModel
    #   'best_complete'       → AF-splice (best conformer, no explicit MODELLER)
    if variant == 'single':
        ai_source = 'no fill (crystal complete)'
    elif variant == 'large_gap_graft' or 'Kabsch' in method or 'Superimposer' in method:
        ai_source = 'BL-Pose (Kabsch graft, no AF)'
    elif variant == 'large_gap_complete' or 'MODELLER' in method:
        ai_source = 'AF-splice + MODELLER'
    elif variant == 'best_complete':
        ai_source = 'AF-splice (best conformer)'
    else:
        ai_source = 'unknown/other'
    n_breaks = manifest.get('backbone_continuity', {}).get('n_breaks', 0)
    n_warnings = len(manifest.get('warnings', []))
    return {
        'pdb_id': pdb_id,
        'n_residues': n_res,
        'MW_kDa_estimate': mw_kda,
        'variant': variant,
        'ai_source': ai_source,
        'n_gaps_grafted': graft.get('n_gaps_grafted', 0),
        'n_gaps_skipped': len(graft.get('skipped', []) or []),
        'n_backbone_breaks': n_breaks,
        'n_warnings': n_warnings,
    }

pdb_ids_all = sorted(p.name for p in DELIVERED_ROOT.iterdir() if p.is_dir())
summary_df = pd.DataFrame([summarise(p) for p in pdb_ids_all])
summary_df


### 3a — KI-Aufteilung (AF-splice vs BL-Pose fallback)

In [ ]:
# KI-Aufteilung: how many proteins used AF-splice vs BL-Pose fallback?
counts = summary_df['ai_source'].value_counts()
print(counts.to_string())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
counts.plot.barh(ax=ax, color=['#2b6cb0', '#c05621', '#a0aec0'][:len(counts)])
ax.set_xlabel('n proteins')
ax.set_title('newbench_27 — AI source split (AF-splice vs BL-Pose fallback)')
for i, (label, v) in enumerate(counts.items()):
    ax.text(v + 0.2, i, f'  {v}', va='center')
plt.tight_layout()
plt.show()


### 3b — Massenaufteilung (molecular-weight histogram)

In [ ]:
# Massenaufteilung: molecular-weight distribution across delivered receptors
import matplotlib.pyplot as plt
mw = summary_df['MW_kDa_estimate'].dropna()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(mw, bins=15, color='#2b6cb0', edgecolor='none')
ax.set_xlabel('Molecular weight (kDa, estimate = 110 Da × n_residues)')
ax.set_ylabel('n proteins')
ax.set_title(f'newbench_27 — mass distribution '
             f'(n={len(mw)}, median={mw.median():.1f} kDa, '
             f'range {mw.min():.1f}–{mw.max():.1f} kDa)')
ax.axvline(mw.median(), color='grey', linestyle=':', linewidth=0.8,
           label=f'median {mw.median():.1f} kDa')
ax.legend()
plt.tight_layout()
plt.show()

# Also show the top-5 largest + smallest
print('Largest by MW:')
print(summary_df.nlargest(5, 'MW_kDa_estimate')[['pdb_id','MW_kDa_estimate','n_residues']].to_string(index=False))
print()
print('Smallest by MW:')
print(summary_df.nsmallest(5, 'MW_kDa_estimate')[['pdb_id','MW_kDa_estimate','n_residues']].to_string(index=False))


### 3c — Actives / inactives ligand sets per target

Scans the usual on-disk locations. If your ligand sets live elsewhere, extend `ACTIVES_CANDIDATE_ROOTS`.

In [ ]:
# Actives / inactives per target — scan for the standard file names.
# Newbench_27 does not (yet) ship ligand sets under delivered/;
# this cell looks in a few known locations and reports what's found.
import pathlib

ACTIVES_CANDIDATE_ROOTS = [
    pathlib.Path('/mnt/netapp1/Store_othcxlwa/DEKOIS-EXP'),
    pathlib.Path('/mnt/netapp1/Store_othcxlwa/newbench_27/actives'),
    pathlib.Path('/mnt/netapp1/Store_othcxlwa/newbench_27/ligands'),
]

def _count_lines(p: pathlib.Path) -> int:
    try:
        return sum(1 for _ in p.open())
    except Exception:
        return 0

def _find_ligand_files(pdb_id: str) -> dict:
    hits = {'actives_smi': None, 'inactives_smi': None,
            'n_actives': 0, 'n_inactives': 0}
    for root in ACTIVES_CANDIDATE_ROOTS:
        if not root.is_dir():
            continue
        # Try lowercase, uppercase, exact-case
        for variant in (pdb_id, pdb_id.lower(), pdb_id.upper()):
            for pattern in ('actives.smi', f'{variant}_actives.smi', 'actives.ism', f'{variant}.ism'):
                p = root / variant / pattern
                if p.is_file():
                    hits['actives_smi'] = str(p); hits['n_actives'] = _count_lines(p); break
            for pattern in ('inactives.smi', f'{variant}_inactives.smi', 'decoys.smi', f'{variant}_decoys.smi'):
                p = root / variant / pattern
                if p.is_file():
                    hits['inactives_smi'] = str(p); hits['n_inactives'] = _count_lines(p); break
    return hits

ligand_rows = []
for pid in pdb_ids_all:
    h = _find_ligand_files(pid)
    h['pdb_id'] = pid
    ligand_rows.append(h)
ligand_df = pd.DataFrame(ligand_rows)[['pdb_id', 'n_actives', 'n_inactives', 'actives_smi', 'inactives_smi']]

n_with = sum(1 for r in ligand_rows if r['n_actives'] > 0 or r['n_inactives'] > 0)
print(f'{n_with} / {len(ligand_rows)} proteins have ligand set data on disk.')
if n_with == 0:
    print(
        'No actives/inactives sets found in the standard locations.\n'
        'If your ligands live elsewhere, extend ACTIVES_CANDIDATE_ROOTS in this cell.'
    )
ligand_df


### 3d — Merged summary table

In [ ]:
# Merged summary table (KI + Masse + actives/inactives) per protein
merged = summary_df.merge(ligand_df, on='pdb_id', how='left')
merged = merged[['pdb_id', 'ai_source', 'MW_kDa_estimate', 'n_residues',
                 'n_gaps_grafted', 'n_gaps_skipped', 'n_backbone_breaks',
                 'n_warnings', 'n_actives', 'n_inactives', 'variant']]
merged


## Step 4 — Loader function

Loads the crystal PDB + FRUTON receptor (with graft) into PyMOL, greys the crystal, colours the filled model sky-blue, and highlights the graft-region residues in hot pink with side-chain sticks.

In [ ]:
# Loader: for a given PDB id, show crystal vs graft in PyMOL,
# highlighting the gap-region residues that FRUTON filled.
import pathlib

def load_newbench_fill(pdb_id: str) -> dict:
    entry = DELIVERED_ROOT / pdb_id
    crystal = BENCH_ROOT / pdb_id / f"{pdb_id}.pdb"
    graft = entry / "graft" / f"{pdb_id}_graft.pdb"
    receptor = entry / "receptor.pdb"
    manifest_path = entry / "graft" / f"{pdb_id}_graft_manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
    return {
        "pdb_id": pdb_id,
        "crystal": crystal if crystal.is_file() else None,
        "graft": graft if graft.is_file() else None,
        "receptor": receptor if receptor.is_file() else None,
        "manifest": manifest,
    }


def pymol_show_fill(pdb_id: str, gray_crystal: bool = True) -> dict:
    r = load_newbench_fill(pdb_id)
    pm.do("reinitialize")
    pm.do("bg_color white")
    if r["crystal"]:
        pm.do(f"load {r['crystal']}, crystal")
        if gray_crystal:
            pm.do("color grey70, crystal")
        pm.do("show cartoon, crystal")
    if r["receptor"]:
        pm.do(f"load {r['receptor']}, filled")
        pm.do("color skyblue, filled")
        pm.do("show cartoon, filled")
    # Highlight graft-region residues in the filled model, if manifest lists them.
    graft_ranges = r["manifest"].get("graft_ranges") or r["manifest"].get("ranges") or []
    for rng in graft_ranges:
        chain = rng.get("chain", "A")
        lo = rng.get("first_resnum") or rng.get("start")
        hi = rng.get("last_resnum") or rng.get("end")
        if lo is None or hi is None:
            continue
        sel = f"filled and chain {chain} and resi {lo}-{hi}"
        pm.do(f"color hotpink, {sel}")
        pm.do(f"show sticks, {sel} and (name CA+CB+N+C+O)")
    if r["crystal"] and r["receptor"]:
        pm.do("align filled and name CA, crystal and name CA")
    pm.do("orient")
    pm.do("zoom")
    print(f"{pdb_id}: crystal={bool(r['crystal'])}, filled={bool(r['receptor'])}, "
          f"n_graft_ranges={len(graft_ranges)}")
    return r


## Step 5 — Interactive selector

In [ ]:
# Interactive selector — pick a PDB and PyMOL updates on the laptop
import ipywidgets as W
from IPython.display import display

dropdown = W.Dropdown(options=pdb_ids, description="PDB:", layout={"width": "260px"})
gray_toggle = W.Checkbox(value=True, description="Grey crystal")
btn = W.Button(description="Show in PyMOL", button_style="primary")
out = W.Output()

def _on_click(_):
    with out:
        out.clear_output()
        pymol_show_fill(dropdown.value, gray_crystal=gray_toggle.value)

btn.on_click(_on_click)
display(W.HBox([dropdown, gray_toggle, btn]), out)


## Optional — Batch walk (saves one `.pse` session per PDB)

In [ ]:
# Optional: walk every delivered newbench_27 protein one at a time,
# saving a session file per PDB so you can re-open later without re-running.
import pathlib
SESSION_DIR = pathlib.Path("~/pymol_sessions_newbench27").expanduser()
SESSION_DIR.mkdir(parents=True, exist_ok=True)

for pid in pdb_ids:
    pymol_show_fill(pid, gray_crystal=True)
    pm.do(f"save {SESSION_DIR / (pid + '.pse')}")
    print("  session ->", SESSION_DIR / (pid + ".pse"))
